Importing and loading artifacts

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import pipeline

# Load config and artifacts
with open("models/best_config.json") as f:
    cfg = json.load(f)

with open("models/best_model_meta.json") as f:
    meta = json.load(f)

vectorizer = joblib.load("models/best_vectorizer.pkl")
model      = joblib.load(meta["file"])

# Load data
train_df = pd.read_csv("Dataset/train.csv")
test_df  = pd.read_csv("Dataset/test.csv")

train_df[cfg["input_field"]] = train_df[cfg["input_field"]].fillna("")
test_df[cfg["input_field"]]  = test_df[cfg["input_field"]].fillna("")

print("Model type :", meta["model_type"])
print("Best F1    :", meta["best_f1"])
print("Train size :", len(train_df))
print("Test size  :", len(test_df))


Model type : logistic_regression
Best F1    : 0.9951
Train size : 33043
Test size  : 11015


Embed training set and Build FIASS Index

In [2]:
print("Vectorizing training set...")
X_train_tfidf = vectorizer.transform(train_df[cfg["input_field"]])

# Convert to dense float32 — FAISS requires dense arrays
X_train_dense = X_train_tfidf.toarray().astype(np.float32)

# Normalize so cosine similarity = dot product (faster FAISS search)
faiss.normalize_L2(X_train_dense)

# Build FAISS flat index (exact nearest neighbor search)
dim   = X_train_dense.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner product = cosine on L2-normalized vectors
index.add(X_train_dense)

print(f"FAISS index built.")
print(f"  Vectors indexed : {index.ntotal}")
print(f"  Vector dimension: {dim}")


Vectorizing training set...
FAISS index built.
  Vectors indexed : 33043
  Vector dimension: 50000


TF-IDF key phrase extractor

In [3]:
def get_top_tfidf_phrases(text, vectorizer, top_n=10):
    vec = vectorizer.transform([text])
    indices = np.argsort(vec.data)[::-1][:top_n]
    feature_names = np.array(vectorizer.get_feature_names_out())
    top_features = feature_names[vec.indices[indices]]
    top_scores   = vec.data[indices]
    return list(zip(top_features, top_scores.round(4)))

# Quick smoke test
sample_text = test_df[cfg["input_field"]].iloc[0]
phrases = get_top_tfidf_phrases(sample_text, vectorizer)
print("Top TF-IDF phrases:")
for phrase, score in phrases:
    print(f"  {phrase:<30} {score}")


Top TF-IDF phrases:
  clark                          0.4629
  minneapolis                    0.2697
  gun                            0.2443
  march                          0.1904
  got gun                        0.1698
  paramedic                      0.161
  lee                            0.141
  black                          0.1169
  protester                      0.1153
  speaker                        0.1106


Human readable explainer model

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer  = AutoTokenizer.from_pretrained("google/flan-t5-large")
llm_model  = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

def generate_explanation(prompt, max_new_tokens=200):
    inputs  = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = llm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Smoke test
print(generate_explanation("Is the sky blue? Answer yes or no and explain briefly."))


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

The sky is blue because it is a color of light. Light is blue because it is reflected off of the surface of the Earth. The answer: yes.


RAG Pipeline

In [13]:
def rag_explain(text, label_pred, top_k=3):
    # 1. Vectorize and L2-normalize query
    vec = vectorizer.transform([text]).toarray().astype(np.float32)
    faiss.normalize_L2(vec)

    # 2. Retrieve top-k similar training examples
    D, I = index.search(vec, top_k)
    retrieved = train_df.iloc[I[0]]

    # 3. Top TF-IDF phrases
    phrases    = get_top_tfidf_phrases(text, vectorizer, top_n=5)
    phrase_str = ", ".join([f"'{p}'" for p, _ in phrases])

    # 4. Neighbor label stats
    neighbor_labels  = retrieved["label"].tolist()
    same_label_count = sum(1 for l in neighbor_labels if l == label_pred)
    label_str        = "FAKE" if label_pred == 0 else "REAL"

    # 5. Model confidence
    confidence = model.predict_proba(vectorizer.transform([text]))[0][label_pred] * 100

    # 6. Template explanation (deterministic, always complete)
    template = (
        f"This article is classified as {label_str} with {confidence:.1f}% confidence. "
        f"The most distinctive phrases found were: {phrase_str}. "
        f"These phrases appear frequently in {label_str} articles in the training data. "
        f"{same_label_count} out of {top_k} most similar training articles were also "
        f"labeled {label_str}, further supporting this prediction."
    )

    # 7. LLM adds a short tone/topic description (small, constrained task)
    snippet     = text[:150].replace("\n", " ")
    llm_prompt  = f"In one short sentence, describe the topic of this news snippet: \"{snippet}\""
    llm_tone    = generate_explanation(llm_prompt, max_new_tokens=60)
    explanation = template + f" Topic summary: {llm_tone}"

    return explanation, phrases, retrieved[["label", cfg["input_field"]]].head(top_k)

# Smoke test
sample_idx  = 1
sample_text = test_df[cfg["input_field"]].iloc[sample_idx]
sample_pred = model.predict(vectorizer.transform([sample_text]))[0]
true_label  = test_df["label"].iloc[sample_idx]

explanation, phrases, neighbors = rag_explain(sample_text, sample_pred)
print(f"True label : {'REAL' if true_label == 1 else 'FAKE'}")
print(f"Prediction : {'REAL' if sample_pred == 1 else 'FAKE'}")
print(f"\nExplanation:\n{explanation}")


True label : REAL
Prediction : REAL

Explanation:
This article is classified as REAL with 99.8% confidence. The most distinctive phrases found were: 'muslimmajority country', 'muslimmajority', 'refugee people', 'order', 'trump'. These phrases appear frequently in REAL articles in the training data. 3 out of 3 most similar training articles were also labeled REAL, further supporting this prediction. Topic summary: reuters highlights day u.s. president trump fights back amid international criticism outrage


Demo

In [14]:
# Pick a mix: some FAKE, some REAL, including at least one misprediction if any
sample_indices = [10, 20, 30, 40, 50]  # Adjust these indices based on your test set
fake_indices = test_df[test_df["label"] == 0].index[:3].tolist()
real_indices = test_df[test_df["label"] == 1].index[:2].tolist()
sample_indices = fake_indices + real_indices

for idx in sample_indices:
    text       = test_df.loc[idx, cfg["input_field"]]
    true_label = test_df.loc[idx, "label"]
    pred       = model.predict(vectorizer.transform([text]))[0]

    explanation, phrases, neighbors = rag_explain(text, pred)

    true_str = "REAL" if true_label == 1 else "FAKE"
    pred_str = "REAL" if pred == 1 else "FAKE"
    match    = "✓" if true_label == pred else "✗ WRONG"

    print("=" * 70)
    print(f"True: {true_str}  |  Predicted: {pred_str}  {match}")
    print(f"Article snippet: {text[:100].replace(chr(10), ' ')}...")
    print(f"\n{explanation}")
    print()


True: FAKE  |  Predicted: FAKE  ✓
Article snippet: another case white man oppressive america holding black man much potential promise angry black life ...

This article is classified as FAKE with 99.9% confidence. The most distinctive phrases found were: 'clark', 'minneapolis', 'gun', 'march', 'got gun'. These phrases appear frequently in FAKE articles in the training data. 3 out of 3 most similar training articles were also labeled FAKE, further supporting this prediction. Topic summary: a black man protests a white man 's sexism

True: FAKE  |  Predicted: FAKE  ✓
Article snippet: guess trump missed stick stone lesson life critic getting blocked twitter tiniest thingstake example...

This article is classified as FAKE with 98.8% confidence. The most distinctive phrases found were: 'trump', 'blocked', 'ben jerry', 'twitter', 'flavor'. These phrases appear frequently in FAKE articles in the training data. 1 out of 3 most similar training articles were also labeled FAKE, further supporti

Summary

In [15]:
print("=" * 70)
print("NOTEBOOK 4 SUMMARY — RAG Explainability")
print("=" * 70)
print("""
OBJECTIVE
---------
Notebook 4 adds explainability to the fake news classifier built in
Notebooks 1-3. Rather than returning a bare prediction, the system
retrieves evidence from the training data and generates a natural
language explanation for each decision.

RAG PIPELINE
------------
1. Vectorize: Input text is transformed using the same TF-IDF vectorizer
   trained in Notebook 2 (50,000 features, bigrams, body-only).

2. Retrieve: FAISS (IndexFlatIP) performs cosine similarity search over
   L2-normalized TF-IDF vectors of all training documents. The top-3
   most similar training articles are retrieved.

3. Extract: The top-5 TF-IDF phrases with highest weight in the input
   are extracted as the most discriminative linguistic features.

4. Explain: A structured template combines the model's confidence score,
   the key phrases, and neighbor label statistics into a human-readable
   explanation. A flan-t5-large LLM adds a brief topic description.

MODEL PERFORMANCE (from Notebook 3)
------------------------------------
  Best model : Logistic Regression (TF-IDF, bigrams, C=10)
  Test F1    : 0.9951
  Test set   : held out before any training or tuning

WHY RAG FOR EXPLAINABILITY?
----------------------------
Traditional classifiers are black boxes — they return a label with no
justification. RAG grounds the explanation in concrete training evidence:
which similar articles share the prediction, and which specific phrases
drove the decision. This makes the system transparent and auditable.
""")


NOTEBOOK 4 SUMMARY — RAG Explainability

OBJECTIVE
---------
Notebook 4 adds explainability to the fake news classifier built in
Notebooks 1-3. Rather than returning a bare prediction, the system
retrieves evidence from the training data and generates a natural
language explanation for each decision.

RAG PIPELINE
------------
1. Vectorize: Input text is transformed using the same TF-IDF vectorizer
   trained in Notebook 2 (50,000 features, bigrams, body-only).

2. Retrieve: FAISS (IndexFlatIP) performs cosine similarity search over
   L2-normalized TF-IDF vectors of all training documents. The top-3
   most similar training articles are retrieved.

3. Extract: The top-5 TF-IDF phrases with highest weight in the input
   are extracted as the most discriminative linguistic features.

4. Explain: A structured template combines the model's confidence score,
   the key phrases, and neighbor label statistics into a human-readable
   explanation. A flan-t5-large LLM adds a brief topic descri